## Surveillance Anomaly Detection


In [1]:
!pip install torch torchvision torchaudio

In [2]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [3]:
import torch
print(torch.backends.mps.is_available())

True


## Frame Feature Extraction using ResNet

This step converts raw video frames into compact feature vectors using a pretrained **ResNet CNN**.

Instead of training directly on images, each frame is passed through ResNet to extract a **2048-dimensional feature representation**. These features capture important visual patterns while reducing data size.

Pipeline:

Frames → Dataset → DataLoader → ResNet → Feature Vectors (2048)

The extracted features are later used to build temporal sequences for **anomaly detection**.

In [4]:
import os
import glob
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# ===============================
# Dataset Class
# ===============================
class FrameDataset(Dataset):
    def __init__(self, folder_path):
        self.image_paths = sorted(glob.glob(os.path.join(folder_path, "*.jpg")))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        return transform(img)

# ===============================
# Feature Extraction Function
# ===============================
def extract_features_from_folder(folder_path, batch_size=128):

    dataset = FrameDataset(folder_path)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    features = []
    total_batches = len(loader)

    print(f"\nExtracting from: {folder_path}")
    print(f"Total images: {len(dataset)} | Batch size: {batch_size}")

    with torch.no_grad():
        for i, batch in enumerate(loader):
            batch = batch.to(device)

            output = resnet(batch)
            output = output.view(output.size(0), -1)  # (B, 2048)

            features.append(output.cpu().numpy())

            progress = (i + 1) / total_batches * 100
            print(f"Progress: {progress:.2f}%", end="\r")

    print("\nDone.")
    return np.concatenate(features, axis=0)

## Train & Validation Feature Extraction

Frames from each shard in the Train and Validation folders are passed through a pretrained **ResNet** to extract **2048-dimensional features**.  
The extracted features are concatenated to form `train_features` and `val_features`, which are later used for anomaly detection training.

In [12]:
TRAIN_ROOT = "Train"
VAL_ROOT = "Validation"

train_features_all = []
val_features_all = []

# -----------------------------
# TRAIN
# -----------------------------
print("\nStarting TRAIN feature extraction...\n")

train_shards = sorted(os.listdir(TRAIN_ROOT))

for idx, folder in enumerate(train_shards):
    full_path = os.path.join(TRAIN_ROOT, folder)
    
    if os.path.isdir(full_path):
        print(f"\nShard {idx+1}/{len(train_shards)}")
        feats = extract_features_from_folder(full_path, batch_size=128)
        train_features_all.append(feats)

train_features = np.concatenate(train_features_all, axis=0)

print("\nTrain features shape:", train_features.shape)


# -----------------------------
# VALIDATION
# -----------------------------
print("\nStarting VALIDATION feature extraction...\n")

val_shards = sorted(os.listdir(VAL_ROOT))

for idx, folder in enumerate(val_shards):
    full_path = os.path.join(VAL_ROOT, folder)
    
    if os.path.isdir(full_path):
        print(f"\nShard {idx+1}/{len(val_shards)}")
        feats = extract_features_from_folder(full_path, batch_size=128)
        val_features_all.append(feats)

val_features = np.concatenate(val_features_all, axis=0)

print("\nValidation features shape:", val_features.shape)

print("\n🎉 Feature extraction completed!")


Starting TRAIN feature extraction...


Shard 2/90

Extracting from: Train/frames-000000
Total images: 10000 | Batch size: 128
Progress: 100.00%
Done.

Shard 3/90

Extracting from: Train/frames-000001
Total images: 10000 | Batch size: 128
Progress: 100.00%
Done.

Shard 4/90

Extracting from: Train/frames-000002
Total images: 10000 | Batch size: 128
Progress: 100.00%
Done.

Shard 5/90

Extracting from: Train/frames-000003
Total images: 10000 | Batch size: 128
Progress: 100.00%
Done.

Shard 6/90

Extracting from: Train/frames-000004
Total images: 10000 | Batch size: 128
Progress: 100.00%
Done.

Shard 7/90

Extracting from: Train/frames-000005
Total images: 10000 | Batch size: 128
Progress: 100.00%
Done.

Shard 8/90

Extracting from: Train/frames-000006
Total images: 10000 | Batch size: 128
Progress: 100.00%
Done.

Shard 9/90

Extracting from: Train/frames-000007
Total images: 10000 | Batch size: 128
Progress: 100.00%
Done.

Shard 10/90

Extracting from: Train/frames-000008
Total images: 

In [6]:
import os
import numpy as np

os.makedirs("saved_features", exist_ok=True)

## Saving Extracted Features

The extracted frame features are saved to disk using NumPy.  
This avoids recomputing ResNet features every time the model is trained and significantly speeds up experimentation.

In [7]:
np.save("saved_features/train_features_full.npy", train_features)

print("✅ Train features saved.")
print("Size (GB):", train_features.nbytes / (1024**3))

NameError: name 'train_features' is not defined

## Saving Validation Features

The extracted features from the validation dataset are saved to disk as a `.npy` file.  
This allows the validation data to be reused during model evaluation without repeating the feature extraction step.

In [15]:
np.save("saved_features/val_features_full.npy", val_features)

print("✅ Validation features saved.")
print("Size (GB):", val_features.nbytes / (1024**3))

✅ Validation features saved.
Size (GB): 2.288818359375


In [8]:
import numpy as np

print("Loading saved features...\n")

train_features = np.load("saved_features/train_features_full.npy")
val_features = np.load("saved_features/val_features_full.npy")

print("Train features shape:", train_features.shape)
print("Validation features shape:", val_features.shape)

print("\n✅ Features loaded successfully!")

Loading saved features...

Train features shape: (862346, 2048)
Validation features shape: (300000, 2048)

✅ Features loaded successfully!


In [10]:
import numpy as np

train_features = np.load("saved_features/train_features_full.npy")
val_features = np.load("saved_features/val_features_full.npy")

print("Loaded:", train_features.shape, val_features.shape)

Loaded: (862346, 2048) (300000, 2048)


normalize train data only

In [9]:
train_mean = train_features.mean(axis=0)
train_std = train_features.std(axis=0) + 1e-8

train_features = (train_features - train_mean) / train_std
val_features = (val_features - train_mean) / train_std

print("Normalization done")

Normalization done


## Dynamic Sequence Dataset create

In [11]:
import torch
from torch.utils.data import Dataset

class DynamicSequenceDataset(Dataset):
    def __init__(self, features, seq_len=16, stride=5):
        self.features = features
        self.seq_len = seq_len
        self.stride = stride
        self.length = (len(features) - seq_len) // stride

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        start = idx * self.stride
        seq = self.features[start:start+self.seq_len]
        return torch.tensor(seq, dtype=torch.float32)

## Create Datasets

In [12]:
SEQ_LEN = 16
STRIDE = 5
BATCH_SIZE = 32

train_dataset = DynamicSequenceDataset(train_features, SEQ_LEN, STRIDE)
val_dataset = DynamicSequenceDataset(val_features, SEQ_LEN, STRIDE)

print("Train sequences:", len(train_dataset))
print("Val sequences:", len(val_dataset))

Train sequences: 172466
Val sequences: 59996


## Creating DataLoaders

DataLoaders are used to efficiently load data in batches during training and validation.

- **train_loader** loads training sequences with shuffling to improve model learning.
- **val_loader** loads validation sequences without shuffling to maintain the original order for evaluation.

Batch processing helps speed up training and allows the model to process multiple sequences at once.

In [13]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

## LSTM Autoencoder for Temporal Anomaly Detection(this is just for experimentation purpose not final modal)


In [14]:
import torch.nn as nn

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=512):
        super().__init__()

        self.encoder = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, input_dim, batch_first=True)

    def forward(self, x):
        encoded, _ = self.encoder(x)
        decoded, _ = self.decoder(encoded)
        return decoded

# TRAINING SETUP

In [15]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = LSTMAutoencoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

## Training Loop (With Validation Loss)

TEST FEATURES EXTRACTION

In [55]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Using Apple GPU (MPS)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("🚀 Using CUDA GPU")
else:
    device = torch.device("cpu")
    print("⚠ Using CPU")

print("Device:", device)

🚀 Using Apple GPU (MPS)
Device: mps


resnet load

In [56]:
import torchvision.models as models

resnet = models.resnet50(weights="IMAGENET1K_V1")
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
resnet = resnet.to(device)
resnet.eval()

print("✅ ResNet loaded on", device)

✅ ResNet loaded on mps


In [57]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

 Test features Extraction (GPU + Stride=5)

In [58]:
import os
import glob
import numpy as np
from PIL import Image
from tqdm import tqdm

def extract_features_from_folder(folder_path, batch_size=128):
    image_paths = sorted(glob.glob(os.path.join(folder_path, "*.jpg")))
    features = []

    print(f"\nExtracting from: {folder_path}")
    print(f"Total images: {len(image_paths)} | Batch size: {batch_size}")

    with torch.no_grad():
        for i in tqdm(range(0, len(image_paths), batch_size)):
            batch_paths = image_paths[i:i+batch_size]
            batch_images = []

            for img_path in batch_paths:
                img = Image.open(img_path).convert("RGB")
                img = transform(img)
                batch_images.append(img)

            batch_images = torch.stack(batch_images).to(device)

            batch_features = resnet(batch_images)
            batch_features = batch_features.view(batch_features.size(0), -1)

            features.append(batch_features.cpu().numpy())

    return np.concatenate(features, axis=0)

In [98]:
TEST_ROOT = "Test"

test_features_all = []

print("🚀 Starting FULL TEST feature extraction...\n")

for folder in sorted(os.listdir(TEST_ROOT)):
    full_path = os.path.join(TEST_ROOT, folder)

    # Skip CSV files
    if os.path.isdir(full_path):
        feats = extract_features_from_folder(full_path, batch_size=128)
        test_features_all.append(feats)

test_features = np.concatenate(test_features_all, axis=0)

print("\n✅ Test features shape:", test_features.shape)

🚀 Starting FULL TEST feature extraction...


Extracting from: Test/frames-000000
Total images: 10000 | Batch size: 128


100%|██████████| 79/79 [00:43<00:00,  1.82it/s]



Extracting from: Test/frames-000001
Total images: 10000 | Batch size: 128


100%|██████████| 79/79 [00:40<00:00,  1.97it/s]



Extracting from: Test/frames-000004
Total images: 10000 | Batch size: 128


...


Extracting from: Test/frames-000160
Total images: 10000 | Batch size: 128


100%|██████████| 79/79 [00:39<00:00,  1.99it/s]



Extracting from: Test/frames-000172
Total images: 4714 | Batch size: 128


100%|██████████| 37/37 [00:20<00:00,  1.77it/s]



✅ Test features shape: (384714, 2048)


save the test features for reuse in diffrent experiments

In [59]:
os.makedirs("saved_features", exist_ok=True)

np.save("saved_features/test_features_full.npy", test_features)

print("✅ Test features saved.")
print("Size (GB):", test_features.nbytes / (1024**3))

✅ Test features saved.
Size (GB): 2.9351348876953125


load & normalize

In [60]:
import numpy as np

# Load
test_features = np.load("saved_features/test_features_full.npy")

# Normalize using TRAIN stats
test_features = (test_features - train_mean) / train_std

print("Test features normalized:", test_features.shape)

Test features normalized: (384714, 2048)


In [62]:
import torch
from torch.utils.data import DataLoader, TensorDataset

test_tensor = torch.tensor(test_sequences, dtype=torch.float32)
test_dataset = TensorDataset(test_tensor)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

compute reconstruction error(move model to gpu as features have been passed through it)

In [63]:
model = model.to(device)
model.eval()

print("Model moved to:", next(model.parameters()).device)

Model moved to: mps:0


labels assign check

In [154]:
!pip install opencv-python

refer this file only for feature extractions->> models have been trained in (surveillance_anomaly_detection.ipynb)